[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/EXAMEN_DL_BOURBAKI/notebooks/05_autoencoder_representation_learning.ipynb)

## Conclusiones — Representation Learning con Autoencoder

El objetivo de esta notebook fue evaluar si una representación aprendida directamente de la trayectoria intradía podía capturar información predictiva adicional a las features cuantitativas diseñadas manualmente.

Se entrenó un autoencoder sobre los 53 retornos intradía preprocesados, utilizando una arquitectura:

**53 → 32 → 16 → 8 → 16 → 32 → 53**

El entrenamiento fue completamente no supervisado: el target `reod` no fue utilizado para aprender la representación. El espacio latente resultante contiene 8 variables que resumen la trayectoria original.

### Resultados

Se evaluaron tres representaciones utilizando el mismo HistGradientBoostingClassifier:

| Representación | Accuracy | Balanced Accuracy | Macro F1 |
|---|---:|---:|---:|
| Quant Features (14) | 0.4687 | 0.4254 | 0.4144 |
| Latent Features (8) | 0.4471 | 0.3982 | 0.3800 |
| **Quant + Latent (22)** | **0.4725** | **0.4294** | **0.4185** |

Las variables latentes por sí solas no superaron a las features cuantitativas. Sin embargo, su combinación produjo el mejor resultado obtenido hasta ahora.

Esto sugiere que la representación aprendida contiene información complementaria sobre la forma de la trayectoria que no está completamente capturada por las estadísticas construidas manualmente.

### Diagnóstico por clase

La mejora del modelo combinado no fue uniforme.

El recall de la clase bajista (`-1`) aumentó de aproximadamente **30.4% a 32.7%**, mientras que la clase neutral permaneció prácticamente estable alrededor de **74%**.

La clase alcista (`+1`) continuó siendo el principal problema del modelo, con un recall de aproximadamente **22.2%**.

Por tanto, el autoencoder aporta una mejora pequeña pero consistente en las métricas globales, principalmente mediante una mejor identificación de trayectorias asociadas con la clase bajista.

### Conclusión

El mejor benchmark obtenido hasta este punto es:

**HistGradientBoosting + 14 Quant Features + 8 Latent Features**

- Accuracy: **47.25%**
- Balanced Accuracy: **42.94%**
- Macro F1: **41.85%**

La principal limitación ya no parece estar en la capacidad del modelo para detectar neutralidad, sino en la separación direccional de las clases `-1` y `+1`, especialmente en la identificación de movimientos alcistas.

No se continúa optimizando la arquitectura del autoencoder en esta etapa, ya que el objetivo del experimento era determinar si la representación latente aportaba información incremental, lo cual quedó demostrado, aunque con una mejora moderada.

# funciones 

In [ ]:
# src/preprocessing.py

import numpy as np
import pandas as pd


TRAIN_END_DAY = 401
VAL_START_DAY = 402
CLIP_LIMIT = 20.0


def temporal_split(df):
    """
    Split temporal definido en 02.

    Train: day 0-401
    Validation: day 402-502
    """

    train_dev = df.loc[
        df["day"] <= TRAIN_END_DAY
    ].copy()

    val_dev = df.loc[
        df["day"] >= VAL_START_DAY
    ].copy()

    return train_dev, val_dev


def add_missingness_features(df, return_cols):
    out = df.copy()

    out["missing_count"] = (
        out[return_cols].isna().sum(axis=1)
    )

    out["missing_share"] = (
        out["missing_count"] / len(return_cols)
    )

    return out


def fit_robust_params(train_df, return_cols):
    """
    Parámetros estimados SOLO con train.
    """

    params = pd.DataFrame({
        "median": train_df[return_cols].median(),
        "q25": train_df[return_cols].quantile(0.25),
        "q75": train_df[return_cols].quantile(0.75),
    })

    params["iqr"] = (
        params["q75"] - params["q25"]
    )

    return params


def transform_returns(
    df,
    return_cols,
    robust_params,
    clip_limit=CLIP_LIMIT
):
    """
    Robust scaling -> clipping -> imputación neutral.
    """

    out = add_missingness_features(
        df,
        return_cols
    )

    for col in return_cols:

        out[col] = (
            out[col] - robust_params.loc[col, "median"]
        ) / robust_params.loc[col, "iqr"]

    out[return_cols] = (
        out[return_cols]
        .clip(-clip_limit, clip_limit)
        .fillna(0.0)
    )

    return out

In [22]:
import pandas as pd 
import numpy as  np 

quant_features = [
    # Path
    "path_return_bps",
    "abs_path_return_bps",
    "return_early_bps",
    "return_middle_bps",
    "return_late_bps",
    "early_late_change_bps",

    # Activity / volatility
    "realized_vol_bps",

    # Path structure
    "path_efficiency",
    "early_vol_share",
    "late_vol_share",
    "vol_concentration",
    "sign_persistence",
    "path_asymmetry",

    # Data availability
    "n_obs",
]

# Final Quant Feature Builder
def build_quant_features(
    train_df,
    val_df,
    return_cols,
    lower_q=0.001,
    upper_q=0.999
):
    """
    Construye las features cuantitativas finales.

    Los límites de winsorización se estiman exclusivamente
    utilizando train y después se aplican sin recalibración
    sobre validation.
    """

    train_out = train_df.copy()
    val_out = val_df.copy()

    # --------------------------------------------------------
    # 1. Winsorización por intervalo aprendida sólo en train
    # --------------------------------------------------------

    lower_bounds = train_df[return_cols].quantile(lower_q)
    upper_bounds = train_df[return_cols].quantile(upper_q)

    train_out[return_cols] = train_out[return_cols].clip(
        lower=lower_bounds,
        upper=upper_bounds,
        axis=1
    )

    val_out[return_cols] = val_out[return_cols].clip(
        lower=lower_bounds,
        upper=upper_bounds,
        axis=1
    )

    # Ventanas temporales
    early_cols = [f"r{i}" for i in range(0, 18)]
    middle_cols = [f"r{i}" for i in range(18, 36)]
    late_cols = [f"r{i}" for i in range(36, 53)]

    first_half_cols = return_cols[:len(return_cols) // 2]
    second_half_cols = return_cols[len(return_cols) // 2:]

    # --------------------------------------------------------
    # 2. Feature engineering
    # --------------------------------------------------------

    for df in [train_out, val_out]:

        # Disponibilidad
        df["n_obs"] = df[return_cols].notna().sum(axis=1)

        # ---- Path / displacement ----

        df["path_return_bps"] = (
            df[return_cols].sum(axis=1, skipna=True)
        )

        df["abs_path_return_bps"] = (
            df["path_return_bps"].abs()
        )

        df["return_early_bps"] = (
            df[early_cols].sum(axis=1, skipna=True)
        )

        df["return_middle_bps"] = (
            df[middle_cols].sum(axis=1, skipna=True)
        )

        df["return_late_bps"] = (
            df[late_cols].sum(axis=1, skipna=True)
        )

        df["early_late_change_bps"] = (
            df["return_late_bps"]
            - df["return_early_bps"]
        )

        # ---- Volatilidad ----

        squared_returns = df[return_cols] ** 2
        total_variation = squared_returns.sum(axis=1, skipna=True)

        df["realized_vol_bps"] = np.sqrt(total_variation)

        # ---- Path efficiency ----

        total_abs_move = (
            df[return_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        df["path_efficiency"] = np.where(
            total_abs_move > 0,
            df["abs_path_return_bps"] / total_abs_move,
            0.0
        )

        # ---- Distribución temporal de volatilidad ----

        early_variation = (
            squared_returns[early_cols]
            .sum(axis=1, skipna=True)
        )

        late_variation = (
            squared_returns[late_cols]
            .sum(axis=1, skipna=True)
        )

        df["early_vol_share"] = np.where(
            total_variation > 0,
            early_variation / total_variation,
            0.0
        )

        df["late_vol_share"] = np.where(
            total_variation > 0,
            late_variation / total_variation,
            0.0
        )

        # ---- Concentración de volatilidad ----

        df["vol_concentration"] = np.where(
            total_variation > 0,
            squared_returns.max(axis=1) / total_variation,
            0.0
        )

        # ---- Persistencia de signo ----

        path_sign = np.sign(df["path_return_bps"])
        signs = np.sign(df[return_cols])

        observed = df[return_cols].notna()
        same_sign = signs.eq(path_sign, axis=0)

        df["sign_persistence"] = np.where(
            df["n_obs"] > 0,
            (same_sign & observed).sum(axis=1) / df["n_obs"],
            np.nan
        )

        # ---- Asimetría temporal ----

        first_activity = (
            df[first_half_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        second_activity = (
            df[second_half_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        total_activity = first_activity + second_activity

        df["path_asymmetry"] = np.where(
            total_activity > 0,
            (second_activity - first_activity) / total_activity,
            np.nan
        )

    return train_out, val_out, lower_bounds, upper_bounds

In [1]:
import os
import sys
import pandas as pd
import numpy as np

ruta_src = os.path.abspath(os.path.join("..", "src"))
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

from feature_engineering import (
    build_quant_features,
    quant_features,
)

from preprocessing import (
    temporal_split,
    fit_robust_params,
    transform_returns,
)

input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

train_dev, val_dev = temporal_split(train)

robust_params = fit_robust_params(
    train_dev,
    return_cols
)

train_processed = transform_returns(
    train_dev,
    return_cols,
    robust_params
)

val_processed = transform_returns(
    val_dev,
    return_cols,
    robust_params
)

### 1. Matriz de entrada del Autoencoder

Como transform_returns() ya ejecuta las decisiones fijadas en 02, tomamos únicamente r0,...,r52:

In [2]:
X_ae_train = train_processed[return_cols].copy()
X_ae_val = val_processed[return_cols].copy()

print("Train shape:", X_ae_train.shape)
print("Validation shape:", X_ae_val.shape)

print("\nNaNs:")
print("Train:", X_ae_train.isna().sum().sum())
print("Validation:", X_ae_val.isna().sum().sum())

print("\nRange:")
print(
    "Train:",
    X_ae_train.min().min(),
    X_ae_train.max().max()
)
print(
    "Validation:",
    X_ae_val.min().min(),
    X_ae_val.max().max()
)

Train shape: (673751, 53)
Validation shape: (169548, 53)

NaNs:
Train: 0
Validation: 0

Range:
Train: -20.0 20.0
Validation: -20.0 20.0


In [3]:
print("Inf train:",np.isinf(X_ae_train.to_numpy()).sum())
print("Inf validation:",np.isinf(X_ae_val.to_numpy()).sum())

Inf train: 0
Inf validation: 0


## 2. Autoencoder para representación latente

Las 53 observaciones intradía contienen la trayectoria temporal completa del activo.
Las features cuantitativas construidas anteriormente resumen esta trayectoria mediante
estadísticas diseñadas manualmente, lo que potencialmente descarta información sobre
su forma y dinámica temporal.

En esta sección se utiliza un autoencoder para aprender una representación comprimida
de la trayectoria sin utilizar el target `reod`.

La arquitectura propuesta comprime:

53 → 32 → 16 → 8 → 16 → 32 → 53

El espacio latente de dimensión 8 será posteriormente evaluado como representación
alternativa para el problema de clasificación.

El autoencoder se entrena exclusivamente con el conjunto de desarrollo y utiliza
validation únicamente para evaluar generalización de la reconstrucción.

In [4]:
# import sys
# !{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install tensorflow

In [5]:
# Construcción del Autoencoder
#Usaremos TensorFlow/Keras:

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

In [6]:
input_dim = len(return_cols)   # 53
latent_dim = 8

# Input
inputs = Input(shape=(input_dim,), name="returns_input")

# Encoder
x = Dense(32, activation="relu")(inputs)
x = Dense(16, activation="relu")(x)

latent = Dense(
    latent_dim,
    activation="linear",
    name="latent"
)(x)

# Decoder
x = Dense(16, activation="relu")(latent)
x = Dense(32, activation="relu")(x)

outputs = Dense(
    input_dim,
    activation="linear",
    name="reconstruction"
)(x)

autoencoder = Model(
    inputs=inputs,
    outputs=outputs,
    name="intraday_autoencoder"
)

encoder = Model(
    inputs=inputs,
    outputs=latent,
    name="intraday_encoder"
)

In [7]:
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse"
)

autoencoder.summary()

Model: "intraday_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ returns_input (InputLayer)      │ (None, 53)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Dense)          │ (None, 53)             │         1,749 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,829 (18.86 KB)

 Trainable params: 4,829 (18.86 KB)

 Non-trainable params: 0 (0.00 B)

### Por qué linear en latent y output

Aquí los inputs pueden ser negativos y positivos ([-20,20]). No queremos que una activación como ReLU en la salida impida reconstruir retornos negativos.

La función objetivo será simplemente:

$$L_{AE} = \frac{1}{53} \sum_{t=0}^{52} (r_t - \hat{r}_t)^2$$

**Importante:** `reod` no entra en ningún sitio. El autoencoder aprende exclusivamente de las trayectorias.


### Entrenamiento

Usaremos early stopping para no fijar arbitrariamente el número de epochs:

In [8]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_ae_train.to_numpy(dtype=np.float32),
    X_ae_train.to_numpy(dtype=np.float32),

    validation_data=(
        X_ae_val.to_numpy(dtype=np.float32),
        X_ae_val.to_numpy(dtype=np.float32)
    ),

    epochs=50,
    batch_size=1024,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 2.1122 - val_loss: 2.0110
Epoch 2/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.9452 - val_loss: 1.9763
Epoch 3/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.9225 - val_loss: 1.9634
Epoch 4/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.9106 - val_loss: 1.9553
Epoch 5/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.9019 - val_loss: 1.9487
Epoch 6/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.8949 - val_loss: 1.9429
Epoch 7/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.8882 - val_loss: 1.9366
Epoch 8/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.8821 - val_loss: 1.9309
Epoch 9/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.8765 - val_loss: 1.9255
Epoch 10/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.8716 - val_loss: 1.9208
Epoch 11/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.8673 - val_loss: 1.9167
Epoch 12/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step

`X = y` porque el objetivo del autoencoder es:

$$(r_0, \dots, r_{52}) \longrightarrow \text{AE} \longrightarrow (\hat{r}_0, \dots, \hat{r}_{52})$$

No estamos intentando predecir `reod`.


In [9]:
print("Epochs:", len(history.history["loss"]))
print("Final train loss:", history.history["loss"][-1])
print("Final val loss:", history.history["val_loss"][-1])
print("Best val loss:", min(history.history["val_loss"]))

Epochs: 50
Final train loss: 1.8010170459747314
Final val loss: 1.8490188121795654
Best val loss: 1.8490188121795654


el autoencoder sí aprendió y generaliza razonablemente, aunque todavía no sabemos si esa representación sirve para reod.

In [10]:
# Extraer representación latente
Z_train = encoder.predict(
    X_ae_train.to_numpy(dtype=np.float32),
    batch_size=2048,
    verbose=0
)

Z_val = encoder.predict(
    X_ae_val.to_numpy(dtype=np.float32),
    batch_size=2048,
    verbose=0
)

latent_cols = [f"z{i}" for i in range(latent_dim)]

Z_train = pd.DataFrame(
    Z_train,
    columns=latent_cols,
    index=train_dev.index
)

Z_val = pd.DataFrame(
    Z_val,
    columns=latent_cols,
    index=val_dev.index
)

print("Latent train:", Z_train.shape)
print("Latent validation:", Z_val.shape)

print("\nNaNs train:", Z_train.isna().sum().sum())
print("NaNs validation:", Z_val.isna().sum().sum())

Z_train.describe()

Latent train: (673751, 8)
Latent validation: (169548, 8)

NaNs train: 0
NaNs validation: 0


,z0,z1,z2,z3,z4,z5,z6,z7
count,673751.000000,673751.000000,673751.000000,673751.000000,673751.000000,673751.000000,673751.000000,673751.000000
mean,-0.048282,-0.284507,-2.393122,-1.051292,-1.256927,-1.935204,3.219166,1.029046
std,1.786697,1.410273,2.528850,2.776299,1.752979,2.508482,1.962990,1.765333
min,-71.150017,-45.105316,-111.393387,-112.036697,-57.371723,-32.826302,-8.211095,-12.722261
25%,-0.448182,-0.524073,-2.853896,-1.167207,-1.581988,-2.805434,2.052047,0.243897
50%,-0.089952,-0.145522,-1.666492,-0.227549,-0.778307,-1.974965,2.711994,0.539881
75%,0.544840,0.252978,-1.072666,0.096155,-0.347329,-1.417992,3.791953,1.271881
max,29.460073,23.924887,11.910680,12.543064,7.989085,77.330215,58.155712,63.741524


In [14]:
#Experimento — Latent features solamente
# Entrenamos el mismo HGB que produjo 46.87%, pero ahora usando únicamente las 8 dimensiones latentes:
y_train = train_dev["reod"]
y_val = val_dev["reod"]
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
hgb_latent = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=31,
    random_state=42
)

hgb_latent.fit(Z_train, y_train)

y_pred_latent = hgb_latent.predict(Z_val)

print("HGB — Latent Features")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_latent):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_latent):.4f}")
print(f"Macro F1:          {f1_score(y_val, y_pred_latent, average='macro'):.4f}")

HGB — Latent Features
Accuracy:          0.4471
Balanced Accuracy: 0.3982
Macro F1:          0.3800


| Features |   Accuracy | Balanced Acc. |   Macro F1 |
| -------- | ---------: | ------------: | ---------: |
| 14 Quant | **46.87%** |    **42.54%** | **41.44%** |
| 8 Latent |     44.71% |        39.82% |     38.00% |


In [16]:
# Reconstruimos exactamente las 14 Quant Features de 04
train_quant, val_quant, _, _ = build_quant_features(
    train_dev,
    val_dev,
    return_cols
)

X_train_quant = train_quant[quant_features].copy()
X_val_quant = val_quant[quant_features].copy()

print("Quant train:", X_train_quant.shape)
print("Quant validation:", X_val_quant.shape)

Quant train: (673751, 14)
Quant validation: (169548, 14)


In [17]:
from sklearn.impute import SimpleImputer

quant_imputer = SimpleImputer(strategy="median")

X_train_quant_imp = quant_imputer.fit_transform(X_train_quant)
X_val_quant_imp = quant_imputer.transform(X_val_quant)

print("NaNs train:", np.isnan(X_train_quant_imp).sum())
print("NaNs validation:", np.isnan(X_val_quant_imp).sum())

NaNs train: 0
NaNs validation: 0


In [18]:
X_train_combined = np.column_stack([
    X_train_quant_imp,
    Z_train.to_numpy()
])

X_val_combined = np.column_stack([
    X_val_quant_imp,
    Z_val.to_numpy()
])

print("Combined train:", X_train_combined.shape)
print("Combined validation:", X_val_combined.shape)

Combined train: (673751, 22)
Combined validation: (169548, 22)


In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

y_train = train_dev["reod"]
y_val = val_dev["reod"]

hgb_combined = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=31,
    random_state=42
)

hgb_combined.fit(X_train_combined, y_train)

y_pred_combined = hgb_combined.predict(X_val_combined)

print("HGB — Quant + Latent")
print("--------------------")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_combined):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_combined):.4f}")
print(f"Macro F1:          {f1_score(y_val, y_pred_combined, average='macro'):.4f}")

HGB — Quant + Latent
--------------------
Accuracy:          0.4725
Balanced Accuracy: 0.4294
Macro F1:          0.4185


In [20]:
# ¿Dónde mejoró Quant + Latent?
from sklearn.metrics import classification_report, confusion_matrix

print("HGB — Quant + Latent")
print(
    classification_report(
        y_val,
        y_pred_combined,
        labels=[-1, 0, 1],
        digits=4
    )
)

HGB — Quant + Latent
              precision    recall  f1-score   support

          -1     0.3904    0.3267    0.3557     52006
           0     0.5489    0.7401    0.6303     71519
           1     0.3444    0.2215    0.2696     46023

    accuracy                         0.4725    169548
   macro avg     0.4279    0.4294    0.4185    169548
weighted avg     0.4448    0.4725    0.4482    169548



In [21]:
cm_combined = confusion_matrix(
    y_val,
    y_pred_combined,
    labels=[-1, 0, 1],
    normalize="true"
)

cm_combined_df = pd.DataFrame(
    cm_combined,
    index=["true_-1", "true_0", "true_1"],
    columns=["pred_-1", "pred_0", "pred_1"]
)

cm_combined_df

,pred_-1,pred_0,pred_1
true_-1,0.326674,0.436334,0.236992
true_0,0.160866,0.740125,0.099009
true_1,0.326467,0.452057,0.221476


Qué hizo realmente el autoencoder?

La mejora global:

46.87%→47.25%

proviene principalmente de que el espacio latente ayuda al modelo a reconocer mejor la clase bajista -1.

Y esto encaja con lo que habíamos encontrado antes: en validation aparecía una estructura bajista más clara que alcista. El autoencoder parece estar capturando parte de esa forma de la trayectoria.

Pero nuestro problema fundamental sigue sin resolverse:

$$Recall_{+1}=22.15% $$

La representación latente aporta información complementaria, pero el incremento es modesto y está concentrado principalmente en mejorar la detección de movimientos bajistas.